# Mission 2 L2 재학습 노트북 (제출용 초안 · main `4bfd2c4`)

- 과제: 발화 1개(startAt~endAt 음성만)로 화자 분류 — 신고자(1) / 119대원(0), 지표 Accuracy
- 모델: `SpeakerCNNLarge` × torch-seed 1/2/3, 소프트보팅 앙상블 1파일
- 데이터: **Training(서울)만 학습**, call_id 80/20 분할(`--seed 42`)로 에폭 선택. 공식 Validation은 **평가만**(backprop·에폭 선택 금지)
- 라벨: 제공된 speaker 라벨 그대로(`verify_mission2_raw_labels.py`로 대조). text 사용·라벨 수정·통화 단위 정보 사용 금지(#41, 운영진 10-01/10-05)
- 위에서 아래로 한 번 실행하면 원본 → 스펙트로그램 → 묶기 → 시드별 학습(에폭별 로그) → 앙상블 → 공식 Val까지 재현됩니다. 모든 출력은 Drive 로그에도 남깁니다.

## 0. 설정

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, time

COMMIT     = "4bfd2c4"                                   # 실행한 main 커밋 (zip 이름과 일치시킬 것)
REPO       = Path("/content/dcc2026-prelim-main-4bfd2c4")  # 압축 푼 코드
DATA_ROOT  = Path("/content/drive/.shortcut-targets-by-id/1Ezl13Cpmt2lGgl-FqISYJWfop8omii6z/대학부 데이터")
MANIFEST   = REPO / "data/manifests/dcc_manifest_v1.csv"  # 32,840행 (Training 29,200 + Val 3,640)
TRAIN_LABEL_ROOT = DATA_ROOT / "Training/2.라벨링데이터"   # 제공 라벨 JSON 폴더 (실제 레이아웃에 맞게 확인)

WORK  = Path("/content/m2_work")                          # 개별 스펙트로그램(~89만 개)은 여기만, Drive 금지
OUT   = Path("/content/drive/MyDrive/dcc2026/m2_l2_rerun_" + COMMIT)  # 묶은 파일·인덱스·ckpt·로그
WORK.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
LOG = OUT / "run_log.txt"

SEEDS       = [1, 2, 3]
EPOCHS      = 30
CHUNK       = 1          # --resume 단위 에폭 수 (#43: 난수 상태까지 이어받음)
USE_AMP     = True       # 첫 에폭 점검 실패 시 False로 바꿔 처음부터
RUN_BUILD   = True       # 스펙트로그램·인덱스·묶기
RUN_TRAIN   = True
RUN_EVAL    = True

# AMP 점검 기준: 기존 fp32 런 seed1 1에폭 값(팀 로그에서 채울 것)
FP32_EPOCH1_REF = {"train_loss": None, "val_acc": None}
AMP_MAX_ACC_DROP = 0.02  # 2%p 넘게 낮으면 중단

os.chdir(REPO); sys.path.insert(0, str(REPO / "src"))
os.environ["DCC_DATA_ROOT"] = str(DATA_ROOT)

def run(cmd):
    # 출력을 화면과 Drive 로그에 동시에 남김
    line = "$ " + " ".join(map(str, cmd))
    print(line)
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(f"\n[{time.strftime('%F %T')}] {line}\n")
        p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             env={**os.environ, "PYTHONUNBUFFERED": "1"})
        for l in p.stdout:
            print(l, end="", flush=True); f.write(l)
        rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"exit {rc}: {line}")

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

## 1. 환경 기록

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
import torch, numpy, pandas
print("commit", COMMIT, "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
assert MANIFEST.exists(), "manifest 없음 — Colab에 만들어 둔 dcc_manifest_v1.csv를 복사"

## 2. 스펙트로그램 (mel 40 / hop 160 / FFT 512 / 8kHz, 발화 구간만)

In [ ]:
TR_DIR, TR_IDX = WORK/"spec_train", WORK/"m2_spectrogram_index_full.parquet"
VA_DIR, VA_IDX = WORK/"spec_val",   WORK/"m2_spectrogram_index_val.parquet"
if RUN_BUILD:
    run([sys.executable, "scripts/build_mission2_spectrograms.py", "--manifest", MANIFEST, "--data-root", DATA_ROOT,
         "--split", "training", "--n-calls", "-1", "--seed", "42", "--workers", os.cpu_count(),
         "--output-dir", TR_DIR, "--index-output", TR_IDX, "--failures-log", OUT/"spec_train_failures.json"])
    run([sys.executable, "scripts/build_mission2_spectrograms.py", "--manifest", MANIFEST, "--data-root", DATA_ROOT,
         "--split", "validation", "--n-calls", "-1", "--seed", "42", "--workers", os.cpu_count(),
         "--output-dir", VA_DIR, "--index-output", VA_IDX, "--failures-log", OUT/"spec_val_failures.json"])
import pandas as pd
tr, va = pd.read_parquet(TR_IDX), pd.read_parquet(VA_IDX)
print("train 발화", len(tr), "통화", tr.call_id.nunique(), "| val 발화", len(va), "통화", va.call_id.nunique())
# 기대: train 약 889,358발화 / 29,200통화, val 111,947발화 / 3,640통화

## 3. 제공 라벨 대조 (라벨 수정 없음, 새로 만든 인덱스는 불일치 0행이 정상)

In [ ]:
TR_IDX_PROVIDED = WORK/"m2_spectrogram_index_full_provided_labels.parquet"
if RUN_BUILD:
    run([sys.executable, "scripts/verify_mission2_raw_labels.py", "--index", TR_IDX,
         "--label-root", TRAIN_LABEL_ROOT, "--output", TR_IDX_PROVIDED])

## 4. 묶기 (float16 1파일 ~7GB → Drive)

In [ ]:
PACKED, PACKED_IDX = OUT/"m2_spectrograms_train_packed.npy", OUT/"m2_spectrogram_index_train_packed.parquet"
if RUN_BUILD:
    run([sys.executable, "scripts/pack_mission2_spectrograms.py", "--index", TR_IDX_PROVIDED,
         "--spectrogram-dir", TR_DIR, "--packed-output", PACKED, "--index-output", PACKED_IDX])
    print("packed sha256", sha256(PACKED))

## 5. 시드별 학습 (에폭별 로그 보존 · 에폭 선택은 Training 내부 held-out만)

In [ ]:
def ckpt(s): return OUT/f"m2_cnn_large_raw_ts{s}.pt"

def train(seed, n_epochs, resume):
    cmd = [sys.executable, "scripts/train_mission2_cnn.py",
           "--index", PACKED_IDX, "--packed-spectrograms", PACKED,
           "--model", "large", "--seed", "42", "--torch-seed", seed, "--test-size", "0.2",
           "--epochs", n_epochs, "--batch-size", "64", "--lr", "1e-3", "--lr-patience", "2",
           "--device", "cuda", "--num-workers", "2", "--model-output", ckpt(seed)]
    if USE_AMP: cmd.append("--amp")
    if resume:  cmd.append("--resume")
    run(cmd)

# 5-1. AMP 점검: seed 1을 1에폭만 돌리고 fp32 기준과 비교 → 실패면 USE_AMP=False로 처음부터
if RUN_TRAIN and USE_AMP and not ckpt(1).exists():
    train(1, 1, resume=False)
    print(">>> 위 로그의 1에폭 train loss / val acc를 FP32_EPOCH1_REF와 비교하세요.")
    print(">>> NaN/inf, GradScaler 스케일 계속 감소, val acc가 fp32보다 2%p 넘게 낮으면 중단.")

In [ ]:
# 5-2. 전체 학습: CHUNK 에폭씩 --resume (런타임 끊기면 이 셀만 다시 실행)
import torch
def done_epochs(s):
    if not ckpt(s).exists(): return 0
    c = torch.load(ckpt(s), map_location="cpu", weights_only=False)
    for k in ("epochs_trained", "epoch", "total_epochs"):
        if isinstance(c, dict) and k in c: return int(c[k])
    raise KeyError("체크포인트에서 학습 에폭 수 키를 확인할 것: " + str(list(c)[:20]))

if RUN_TRAIN:
    for s in SEEDS:
        while (e := done_epochs(s)) < EPOCHS:
            train(s, min(CHUNK, EPOCHS - e), resume=e > 0)
        print(f"seed {s} 완료 · sha256 {sha256(ckpt(s))}")

## 6. 앙상블 1파일 (`--ckpt_path` 1개 규칙)

In [ ]:
ENS = OUT/"m2_cnn_large_raw_ensemble.pt"
run([sys.executable, "scripts/pack_mission2_l2_ensemble.py"] +
    sum([["--ckpt-path", ckpt(s)] for s in SEEDS], []) + ["--output", ENS])
SHAS = {f"ts{s}": sha256(ckpt(s)) for s in SEEDS} | {"ensemble": sha256(ENS)}
json.dump({"commit": COMMIT, "amp": USE_AMP, "sha256": SHAS}, open(OUT/"ckpt_sha256.json", "w"), indent=2)
SHAS

## 7. 공식 Validation (평가만 · 발화 하나씩)

In [ ]:
RES = {}
if RUN_EVAL:
    for name, p in [(f"ts{s}", ckpt(s)) for s in SEEDS] + [("ensemble", ENS)]:
        log, pred = OUT/f"val_{name}.json", OUT/f"val_{name}_pred.csv"
        run([sys.executable, "scripts/evaluate_mission2_l2_on_official_validation.py", "--ckpt-path", p,
             "--index", VA_IDX, "--spectrogram-dir", VA_DIR, "--model", "large", "--device", "cuda",
             "--log-output", log, "--predictions-output", pred])
        RES[name] = json.load(open(log))
    for k, v in RES.items(): print(k, v.get("accuracy"))

## 8. 슬라이스 (분석용 · 모델 입력 아님)

In [ ]:
import numpy as np
pred = pd.read_csv(OUT/"val_ensemble_pred.csv")
df = pred.merge(va, on=["call_id", "utt_idx"], how="left", suffixes=("", "_idx"))
s_col = next(c for c in ["startAt", "start", "start_sec"] if c in df)
e_col = next(c for c in ["endAt", "end", "end_sec"] if c in df)
df["dur"] = df[e_col] - df[s_col]

def overlap_ratio(g):
    st, en = g[s_col].to_numpy(), g[e_col].to_numpy(); r = np.zeros(len(g))
    for i in range(len(g)):
        ov = np.clip(np.minimum(en[i], en) - np.maximum(st[i], st), 0, None); ov[i] = 0
        r[i] = ov.max() / max(en[i] - st[i], 1e-9)
    return pd.Series(r, index=g.index)
df["ov"] = df.groupby("call_id", group_keys=False).apply(overlap_ratio)
df["ok"] = df["speaker"] == df["predicted"]
p90 = df["ov"].quantile(0.9)
slices = {"all": df, f"overlap>=p90({p90:.2f})": df[df.ov >= p90], "<0.5s": df[df.dur < 0.5],
          "0.5-1s": df[(df.dur >= 0.5) & (df.dur < 1)], "<1s": df[df.dur < 1]}
rows = []
for k, d in slices.items():
    rows.append({"slice": k, "n": len(d), "acc": d.ok.mean(),
                 "recall_0": d[d.speaker == 0].ok.mean(), "recall_1": d[d.speaker == 1].ok.mean()})
SL = pd.DataFrame(rows); SL.to_csv(OUT/"val_ensemble_slices.csv", index=False); SL

## 9. 판정 · 환경 고정

- **채택 기준**: 공식 Val이 L1 보험 제출본(84.26%)보다 +3%p 이상이고 에폭별 로그가 완비되면 채택한다.
- 이전 L2 앙상블 91.55% ±0.3%p는 **재현 여부 참고용**이며 채택 기준이 아니다.

In [ ]:
acc = RES["ensemble"]["accuracy"]
L1_VAL_ACC = 0.8426   # M2-L1-9k 공식 Val (보험 제출본, PR #46)
MIN_GAIN   = 0.03     # 채택: L1보다 +3%p 이상 + 에폭별 로그 완비
REF_ACC    = 0.9155   # 이전 L2 앙상블 공식 Val (재현 여부 참고용)

adopt = acc >= L1_VAL_ACC + MIN_GAIN
reproduced = abs(acc - REF_ACC) <= 0.003
verdict = "채택 후보(L1 대비 +{:.2f}%p)".format((acc - L1_VAL_ACC) * 100) if adopt else "채택 보류 — L1 보험 제출본 유지, 로그 공유"
print(f"앙상블 공식 Val {acc:.4%} → {verdict}")
print(f"재현 여부(참고): 이전 91.55% ±0.3%p {'안' if reproduced else '밖'} ({(acc - REF_ACC) * 100:+.2f}%p)")
json.dump({"commit": COMMIT, "amp": USE_AMP, "accuracy": acc, "verdict": verdict,
           "adopt": adopt, "l1_val_acc": L1_VAL_ACC, "min_gain": MIN_GAIN,
           "reproduced_within_0.3pp": reproduced, "ref_acc": REF_ACC,
           "per_seed": {k: v.get("accuracy") for k, v in RES.items()}, "sha256": SHAS},
          open(OUT/"m2_l2_rerun_summary.json", "w"), indent=2, ensure_ascii=False)
run([sys.executable, "-m", "pip", "freeze"])  # Drive 로그에 버전 기록
subprocess.run(f"{sys.executable} -m pip freeze > '{OUT}/pip_freeze.txt'", shell=True)